# 學生學習與作業版｜任務 2：檢查資料品質與日期

**姓名：**蔡東諺　**學號／組別：**C114118124**完成日期：**11/19

**學習目標：** 辨認缺失、重複、異常與含歧義的日期，保留原始紀錄。

本單元與教師版「任務 2」相同標題、資料、教學順序與範例；除了明確標示的作業與填空，其餘教師版教材內容保留。對照重點：程式填空對應教師版「2.2 異常值」。

### 如何完成
1. 先閱讀觀念與執行資料準備；各單元可以獨立啟動，不需要依賴上一單元的變數。
2. **💻 P 作業**：在指定程式 cell 補上 `____`，可先用提供的 Vibe Coding 提示詞詢問 AI。補完再執行，並執行後面的驗證 cell。未填時出現 `NameError` 是預期現象，先完成填空再繼續。
3. **📝 Q 作業**：查詢 AI、比較教材內容，填寫緊接的文字作答區；請記錄實際提示詞、自己的解釋與查證方式。
4. **🎯 M 小任務**：在任務 4、8、12、16 將前面幾單元整合成可執行的成果。不要只交 AI 回覆，要交實際程式、輸出與自己的解釋。
5. 把 Notebook 存到自己的 Google 雲端硬碟，完成後下載／依教師指示繳交 `.ipynb`；小任務保留輸出、文字與 AI 使用紀錄。

**案例限制：**本資料是公式生成的 60 筆模擬紀錄，不能推論真實學生或因果。缺失與 15 小時值在計分後加入。預測時欄位可得性是假設；全部資料已在前半段探索，後續切出的測試集僅示範流程，不能視為完全獨立的新資料。

**完成檢核：**本單元有 1 項程式填空、2 項 AI 查詢作答；若是任務 4、8、12、16，另外有 1 項整合小任務。請保留自己的輸出、回答和驗證紀錄。


## 共用資料準備
以下在記憶體重建與既有教材相同的資料。執行後可以直接開始本單元；不會覆寫 CSV。


In [ ]:
# 匯入套件：numpy 建立隨機資料，pandas 管理表格，pyplot 繪圖
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 固定種子，並維持呼叫順序，以重現原教材的 60 筆模擬資料
rng = np.random.default_rng(42)
n = 60
departments = ["資訊管理", "企業管理", "金融管理"]
# 每個字典鍵是一個欄位；對應的值是 60 名學生的資料
data = {
    "student_id": range(1, n + 1),
    "department": rng.choice(departments, n),
    "sleep_hours": rng.normal(6.7, 1.1, n).round(1),
    "screen_time_hours": rng.normal(5.0, 1.5, n).clip(1, 10).round(1),
    "study_hours": rng.normal(2.5, 1.0, n).clip(0.5, 6).round(1),
    "attendance_rate": rng.normal(88, 8, n).clip(60, 100).round(0),
}
df = pd.DataFrame(data)
# 成績依設定公式與隨機誤差建立；關係來自模擬規則，不能視為因果發現
noise = rng.normal(0, 5, n)
score = (50 + 3 * df["study_hours"] + 2 * df["sleep_hours"]
         - 1.2 * df["screen_time_hours"] + 0.15 * df["attendance_rate"] + noise)
df["quiz_score"] = score.clip(0, 100).round(0)
# 故意在計分後加入資料品質問題；無法從表格還原計分時原始值
df.loc[[5, 17], "sleep_hours"] = np.nan
df.loc[8, "screen_time_hours"] = 15.0
print("模擬資料形狀（列, 欄）：", df.shape)


模擬資料形狀（列, 欄）： (60, 7)


## 2.1 缺失與重複
**觀念說明：**缺失表示未知，不能當成 0。重複整列與識別碼重複是兩件事；先確認業務意義再決定是否刪除。
**預測：**睡眠欄位有幾筆缺失？其他欄位如何？


In [ ]:
# isna() 找缺失；sum() 在每欄加總 True 的數量
print(df.isna().sum())
print("整列完全重複：", df.duplicated().sum())
print("學生編號重複：", df["student_id"].duplicated().sum())
# 顯示含睡眠缺失的原始列，先觀察而不是直接刪除
display(df.loc[df["sleep_hours"].isna()])


student_id           0
department           0
sleep_hours          2
screen_time_hours    0
study_hours          0
attendance_rate      0
quiz_score           0
dtype: int64
整列完全重複： 0
學生編號重複： 0


,student_id,department,sleep_hours,screen_time_hours,study_hours,attendance_rate,quiz_score
5,6,金融管理,NaN,2.8,4.3,96.0,92.0
17,18,資訊管理,NaN,4.5,2.9,100.0,82.0


## 📝 學生作業 02-Q1｜AI 查詢與作答：未知與缺失
**任務：**「未知」、0、NaN 有什麼差異？未確認日期為何不能直接猜？

**可使用的 AI 提示詞：**
```text
我是 Python 與資料分析初學者。請比較未知、數值零、缺失值，並解釋 01-02-2026 在日月格式不明時應怎麼處理。
請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
```
先閱讀教材，再詢問 AI。請記錄工具名稱、實際提示詞；若 AI 提供資料來源，記錄可查證的網址／文件名稱與日期，並自行檢查，無法查證時寫「尚未查證」。不要提供真實個資。


### ✍️ 作業 02-Q1 專用作答區：請雙擊此文字儲存格填寫
- AI 工具及實際提示詞：chatbox，我是 Python 與資料分析初學者。請比較未知、數值零、缺失值，並解釋 01-02-2026 在日月格式不明時應怎麼處理。
請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
- AI 回答重點（請自行整理）：未知

是你自己放進去的標記（如 "未知"、"N/A"、""）。
用字串比對找出它：df[col]=="未知"、df[col].isin(["未知","N/A"])、df[col]==""。
數值零

是真的等於 0（或 0.0），可以直接參加加減、平均等運算。
偵測：df[col]==0 或 (df[col]==0).sum()。
缺失值

pandas 內部用 NaN（float）、None（object）或 pd.NA 表示「應該有值但目前沒有」。
偵測：pd.isna(df[col]) 或 df[col].isna()。許多統計函式會自動把它當作缺失處理。
日期字串 01-02-2026（日月不明）

可能是 DD‑MM‑YYYY（01 日 02 月）或 MM‑DD‑YYYY（02 日 01 月）。
轉換時要 顯式告訴 pandas 哪個部份是日或月：
dayfirst=True → 把第一個數字當日（DD‑MM‑YYYY）
dayfirst=False（預設） → 把第一個數字當月（MM‑DD‑YYYY）
format="%d-%m-%Y" → 明確說是 DD‑MM‑YYYY
加上 errors="coerce" 會把無法解析的變成 NaT（時間的缺失值），之後用 pd.isna() 檢查哪些失敗。
- 我的理解與具體例子：未知、數值零、缺失值 在 pandas 中有不同的內部表示與偵測方式：未知通常是你自己放入的標記字串或特殊數值，數值零是真實的量，缺失值則是 pandas 內建的 NaN/None/pd.NA，必須用 pd.isna() 來偵測。
日期字串 01-02-2026 在日月格式不明時，不要依賴預設猜測；應該要顯式指定 dayfirst、yearfirst 或 format 參數，並使用 errors='coerce' 把無法解析的變成 NaT，之後再用 isna().sum() 檢查哪些仍未成功轉換。
-
import pandas as pd
df = pd.read_csv('sample.csv')
##### #未知偵測
print("未知備註數:", (df['備註'] == '未知').sum())
##### # 零偵測
print("study_hours 為 0 的筆數:", (df['study_hours'] == 0).sum())
##### # 缺失值偵測
print("備註缺失值數:", df['備註'].isna().sum())
##### # 日期處理（示範 dayfirst=True）
df['date'] = pd.to_datetime(df['日期字串'], dayfirst=True, errors='coerce')
print("日期轉換失敗筆數:", df['date'].isna().sum())
print(df[['日期字串','date']].head())

- 查證來源／方式及日期（無法查證請註明）：資料來源：

pandas 官方文件《Working with missing data》（https://pandas.pydata.org/docs/user_guide/missing_data.html）— 說明 NaN、None、pd.NA 的定義，以及 .mean() 預設 skipna=True 忽略缺失值的計算邏輯。

pandas 官方文件 pandas.to_datetime() 說明（https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html）— 說明透過 format 參數明確指定 %m-%d-%Y 與 %d-%m-%Y 解析日期字串的規範。

國際標準化組織 ISO 8601 日期與時間表示法標準（YYYY-MM-DD 無歧義格式）。

驗證方式：

數值與缺失值運算測試：在 Colab 建立包含 4.0、0.0 與 np.nan 的測試資料，分別執行 .mean() 與 .fillna(0).mean()，確認 0 會計入分母、而 NaN 預設被忽略的行為差異。

日期格式解析測試：對字串 "01-02-2026" 分別傳入 format="%m-%d-%Y" 與 format="%d-%m-%Y" 執行 pd.to_datetime()，實際驗證兩者會轉換出 2026-01-02 與 2026-02-01 兩種不同日期，確認未指定格式會產生歧義。

驗證日期：2026-11-12
- 我檢查或修正 AI 回答的地方：無


## 2.2 異常值需要依規格判斷
15 小時是在產生成績之後故意置入，超過本案例原始設定的 10 小時上限；真實生活中不能僅憑超過 10 小時就認定錯誤。先標記、查證與記錄。


## 💻 學生作業 02-P1｜Vibe Coding 程式填空
**對應教師版：**任務 2「2.2 異常值」的示範程式。

**任務：**建立布林篩選條件 flag，判斷 screen_time_hours 是否大於 10；請勿刪除或修改資料。

**執行方式：**找到下一個程式儲存格中的 `____`，使用自己撰寫或 AI 建議的程式補齊，保留其他已提供的程式與註解。補完後先執行該 cell，再執行緊接的自動檢查；後續單元內容可能依賴此變數。

**可使用的 Vibe Coding 提示詞：**
```text
我是 Python 與資料分析初學者，正在 Google Colab 使用 pandas、numpy、matplotlib 或 scikit-learn。
我的任務是：建立布林篩選條件 flag，判斷 screen_time_hours 是否大於 10；請勿刪除或修改資料。
請依我貼上的上下文，只協助補上 ____，保留指定變數名稱與後續程式，不修改原始 df。
請解釋每一步的作用，提供繁體中文註解及可手動驗證的方法。
不要虛構執行結果；如需額外資訊，先指出缺少什麼。
```
完成後記錄實際使用的提示詞與自己改動的地方。


In [ ]:
# 根據本案例原先生成規格標記高於 10 小時的紀錄
flag = df["screen_time_hours"]>10 # TODO：找出本案例超過 10 小時的資料
print("超過案例生成上限的筆數：", flag.sum())
display(df.loc[flag, ["student_id", "screen_time_hours", "quiz_score"]])


超過案例生成上限的筆數： 1


,student_id,screen_time_hours,quiz_score
8,9,15.0,80.0


### ✅ 作業 02-P1｜執行驗證
執行下方程式，並同時檢查上方實際輸出。自動檢查只涵蓋主要條件；如果圖表、說明或資料來源不符任務，仍需人工修正。


In [ ]:
assert flag.equals(df["screen_time_hours"].gt(10))
assert len(flag) == len(df)
print("檢查通過：篩選條件正確。")


檢查通過：篩選條件正確。


### ✍️ 作業 02-P1｜我的完成紀錄
- 我實際使用的提示詞（若未使用 AI 請註明）：我是 Python 與資料分析初學者，正在 Google Colab 使用 pandas、numpy、matplotlib 或 scikit-learn。
我的任務是：建立布林篩選條件 flag，判斷 screen_time_hours 是否大於 10；請勿刪除或修改資料。
請依我貼上的上下文，只協助補上 ____，保留指定變數名稱與後續程式，不修改原始 df。
請解釋每一步的作用，提供繁體中文註解及可手動驗證的方法。
不要虛構執行結果；如需額外資訊，先指出缺少什麼。
- 我補上的程式在做什麼：找出本案例超過 10 小時的資料
- 執行或驗證遇到的問題與修正：無
- 檢查結果及我如何確認：程式輸出成功，並與老師對照


## 📝 學生作業 02-Q2｜AI 查詢與作答：品質問題
**任務：**完整資料列重複與 student_id 重複有何不同？舉一個不能直接刪除的情境。

**可使用的 AI 提示詞：**
```text
我是 Python 與資料分析初學者。請舉例比較完整資料列重複與學生編號重複，並說明為何需先確認資料記錄意義。
請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
```
先閱讀教材，再詢問 AI。請記錄工具名稱、實際提示詞；若 AI 提供資料來源，記錄可查證的網址／文件名稱與日期，並自行檢查，無法查證時寫「尚未查證」。不要提供真實個資。


### ✍️ 作業 02-Q2 專用作答區：請雙擊此文字儲存格填寫
- AI 工具及實際提示詞：gemini，我是 Python 與資料分析初學者。請舉例比較完整資料列重複與學生編號重複，並說明為何需先確認資料記錄意義。
請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
- AI 回答重點（請自行整理）：完整資料列重複」＝ 複製貼上錯誤（直接刪）

特徵：整列所有欄位都一模一樣（同人、同天、同分數）。

處理：這通常是系統重送或手誤複製造成的「髒資料」，可以直接保留一筆、把多餘的刪掉。

「學生編號重複」＝ 同一人的多筆紀錄（先別刪）

特徵：只有學號相同，但日期或分數不同（例如同一人考了兩次試）。

處理：這極可能是正常的「多筆事件紀錄」，絕對不能隨便刪除，必須先確認分析目的（例如要算平均分、還是只看最新成績）再決定怎麼處理。
- 我的理解與具體例子：整列一樣多半是「系統誤植」，只有學號一樣多半是「多次紀錄」；動手刪除前，一定要先搞清楚「每一列代表一個人，還是代表一次事件」
-
import pandas as pd

##### #建立模擬資料：
##### #第 0、1 列是「完整資料列重複」（同日期、同分數、同學號，純粹重複登錄）
##### #第 2 列是「學生編號重複」（S001 在另一天考了第二次試，分數不同）
df_test = pd.DataFrame({
    "student_id": ["S001", "S001", "S001", "S002"],
    "quiz_date": ["2026-11-01", "2026-11-01", "2026-11-08", "2026-11-01"],
    "quiz_score": [85, 85, 92, 78]
})

print("=== 原始資料表 ===")
print(df_test)

##### # 1. 檢查「完整資料列重複」（預設檢查所有欄位）
print("\n=== 1. 完整資料列是否重複 (df.duplicated()) ===")
print(df_test.duplicated())

##### # 2. 檢查「學生編號重複」（只看 student_id 欄位）
print("\n=== 2. 學生編號是否重複 (df.duplicated(subset=['student_id'])) ===")
print(df_test.duplicated(subset=["student_id"]))

##### # 3. 比較兩種刪除方式的結果差異
print("\n=== 3A. 只刪除「完整資料列重複」後的結果（保留不同日期的考試） ===")
print(df_test.drop_duplicates())

print("\n=== 3B. 若未確認意義，直接依「student_id」去重後的結果（會誤刪第二次考試） ===")
print(df_test.drop_duplicates(subset=["student_id"]))
- 查證來源／方式及日期（無法查證請註明）：資料來源：

pandas 官方文件 pandas.DataFrame.duplicated() 說明（https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html）— 說明預設檢查整列所有欄位（subset=None）與指定特定鍵值欄位（subset=['student_id']）的重複判定邏輯。

pandas 官方文件 pandas.DataFrame.drop_duplicates() 說明（https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html）— 說明依據不同欄位子集刪除重複列時的行為與資料保留規則。

資料倉儲與資料清理基礎觀念（資料粒度 Data Grain 定義）— 說明分析前必須確認資料表「每一列代表一個實體或一筆事件」，才能正確判斷主鍵重複是否合理。

驗證方式：

程式對照測試：在 Google Colab 中建立包含「整列完全相同（同人、同日、同分）」與「僅學號相同但日期、分數不同」的測試資料表，分別執行 df.duplicated() 與 df.duplicated(subset=["student_id"])，確認兩者標記為 True 的列數差異。

刪除影響驗證：分別執行 df.drop_duplicates() 與 df.drop_duplicates(subset=["student_id"])，親自比對輸出結果，確認未確認資料意義就依學號去重會導致同一位學生的後續測驗紀錄被誤刪。

驗證日期：2026-11-12
- 我檢查或修正 AI 回答的地方：無


## 2.3 日期格式與未知值：獨立銷售情境
**觀念說明：**`01-02-2026` 在日／月與月／日規則下可能是不同日期，不能猜。以下例子只轉換明確的 ISO 形式，以及日月數字相同、沒有日月歧義的 `01-01-2026`；「未知」、其他格式先保留為待查。

**提問：**`NaT` 是什麼？它是否表示 0 或可自行補成今天？


In [ ]:
# 四筆小資料專門用來示範日期品質，與學生資料主線分開
sales = pd.DataFrame({"sale_id": [1, 2, 3, 4],
                      "sale_date_raw": ["2026/01/01", "01-01-2026", "未知", "01-02-2026"],
                      "amount": [500, 800, 300, 600]})
sales_clean = sales.copy()  # 不改動原始資料
sales_clean["sale_date_clean"] = pd.NaT  # 新欄先設為無法確認
# 使用精確格式解析 YYYY/MM/DD；錯誤值標成 NaT，保留原值供追查
iso = sales_clean["sale_date_raw"].str.fullmatch(r"\d{4}/\d{2}/\d{2}")
sales_clean.loc[iso, "sale_date_clean"] = pd.to_datetime(
    sales_clean.loc[iso, "sale_date_raw"], format="%Y/%m/%d", errors="coerce")
# 範例中的 01-01-2026 無日月歧義；只處理這個已確認值
same_day_month = sales_clean["sale_date_raw"].eq("01-01-2026")
sales_clean.loc[same_day_month, "sale_date_clean"] = pd.to_datetime(
    sales_clean.loc[same_day_month, "sale_date_raw"], format="%d-%m-%Y", errors="coerce")
# 任何尚未確認的資料都保留原值，列入人工查證清單
sales_clean["needs_review"] = sales_clean["sale_date_clean"].isna()
display(sales_clean)
print("待查筆數：", sales_clean["needs_review"].sum())
assert sales_clean.loc[3, "needs_review"]  # 有歧義日期沒有被任意轉換


,sale_id,sale_date_raw,amount,sale_date_clean,needs_review
0,1,2026/01/01,500,2026-01-01,False
1,2,01-01-2026,800,2026-01-01,False
2,3,未知,300,NaT,True
3,4,01-02-2026,600,NaT,True


待查筆數： 2


## 延伸示範：彙整品質檢查紀錄
缺失、重複與超出範圍是不同問題，處理前應先有可追查的摘要。以下範圍上限只適用模擬資料生成規格。

**驗證提問：**資料筆數是否變少？為何 `sleep_hours` 的缺失不能直接補 0？日期未能轉換時，應保留原文，並洽資料來源確認規則。


In [ ]:
# 先彙整品質數量，不直接變更或刪除資料
quality_summary = pd.Series({
    "rows": len(df),
    "sleep_missing": int(df["sleep_hours"].isna().sum()),
    "duplicate_rows": int(df.duplicated().sum()),
    "duplicate_ids": int(df["student_id"].duplicated().sum()),
    "screen_above_10": int((df["screen_time_hours"] > 10).sum())})
display(quality_summary)
assert quality_summary["rows"] == 60


rows               60
sleep_missing       2
duplicate_rows      0
duplicate_ids       0
screen_above_10     1
dtype: int64

## 本單元學習收束與 AI 使用紀錄

請學生先用自己的話解釋一個關鍵觀念，再指出一個執行結果及驗證方式。記錄：使用的提示詞、AI 回答中檢查或修改的地方、遇到的問題、修正後的證據與本資料限制。

**通用提示詞**：我是 Python 初學者，請逐步協助我完成上面的任務；每段程式加繁體中文註解，說明輸入、處理與輸出，提供兩項可實際執行的驗證，保留原始資料，不要虛構執行結果。
